Collecting milestone 2 trained models and milestone 1 processed test data to start distributed prediction and aggregation for milestone 3.

In [ ]:

import json
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import load_npz

from dask.distributed import Client, as_completed

np.random.seed(42)

ROOT = Path(".")
TEST_NPZ = ROOT / "X_test_normalized.npz"
TEST_META = ROOT / "test_meta.csv"
MODELS_DIR = ROOT / "models_node_level"
MODEL_REGISTRY = MODELS_DIR / "model_registry.csv"
NODE_SUMMARY = MODELS_DIR / "node_training_summary.csv"
MANIFEST_M3 = ROOT / "milestone3_manifest.json"

required = [TEST_NPZ, TEST_META, MODELS_DIR, MODEL_REGISTRY, NODE_SUMMARY]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required milestone files:\n" + "\n".join(missing) +
        "\nRun milestone1.ipynb and milestone2.ipynb fully before milestone3."
    )

print("Milestone 3 prerequisites found.")
print(f"Test features file    : {TEST_NPZ}")
print(f"Test metadata file    : {TEST_META}")
print(f"Model registry file   : {MODEL_REGISTRY}")
print(f"Node summary file     : {NODE_SUMMARY}")


Loading sparse test features, metadata, and verified model registry to prepare inference inputs and aggregation tracking structures.

In [ ]:

X_test_sparse = load_npz(TEST_NPZ).tocsr()
df_test_meta = pd.read_csv(TEST_META)
node_summary_df = pd.read_csv(NODE_SUMMARY)
registry_df = pd.read_csv(MODEL_REGISTRY)

if "integrity" in registry_df.columns:
    trained_rows = registry_df[registry_df["integrity"] == "OK"].copy()
else:
    trained_rows = registry_df.copy()

if trained_rows.empty:
    raise ValueError("No verified trained model rows found in model_registry.csv")

if "row_id" in df_test_meta.columns:
    row_ids = df_test_meta["row_id"].astype(str).tolist()
else:
    row_ids = [str(i) for i in range(X_test_sparse.shape[0])]

print(f"Test feature matrix shape : {X_test_sparse.shape}")
print(f"Test metadata rows        : {len(df_test_meta)}")
print(f"Verified model entries    : {len(trained_rows)}")
print(f"Expected trained labels   : {int(node_summary_df['trained_labels'].sum())}")


Connecting to the live Dask scheduler and scattering test features once so worker nodes can run parallel inference efficiently.

In [ ]:

SCHEDULER_IP = "192.168.0.103"  # replace if needed
SCHEDULER_PORT = 8786

print(f"Connecting to scheduler at tcp://{SCHEDULER_IP}:{SCHEDULER_PORT} ...")
client = Client(f"tcp://{SCHEDULER_IP}:{SCHEDULER_PORT}")

scheduler_info = client.scheduler_info()
active_workers = list(scheduler_info["workers"].keys())

if not active_workers:
    raise RuntimeError("No active workers connected to scheduler. Start worker nodes first.")

print(f"Connected workers: {len(active_workers)}")
for w in active_workers:
    print(f"  {w}")
print(f"Dashboard: {client.dashboard_link}")

print("Scattering test features to workers...")
X_test_ref = client.scatter(X_test_sparse, broadcast=True)
print("Test matrix scattered successfully.")


Collecting model payload bytes from milestone 2 artifacts and preparing label batches for distributed prediction dispatch.

In [ ]:

def parse_label_id(label_name: str) -> int:
    s = str(label_name)
    if s.startswith("label_"):
        return int(s.split("_")[1])
    return int(s)

model_payload_items = []
missing_model_paths = []

for _, row in trained_rows.iterrows():
    label_col = str(row["label"])
    model_path = Path(row["model_path"])
    if not model_path.exists():
        missing_model_paths.append(str(model_path))
        continue
    with open(model_path, "rb") as f:
        model_bytes = f.read()
    model_payload_items.append((label_col, parse_label_id(label_col), model_bytes))

if missing_model_paths:
    print(f"Warning: missing model files: {len(missing_model_paths)}")
    print("First few missing paths:")
    print(missing_model_paths[:5])

if not model_payload_items:
    raise ValueError("No model payloads available for inference.")

model_payload_items = sorted(model_payload_items, key=lambda x: x[1])
print(f"Model payloads prepared: {len(model_payload_items)}")
print(f"First labels: {[x[0] for x in model_payload_items[:5]]}")


Implementing worker-side inference and returning compact batch outputs for threshold aggregation and fallback label selection.

In [ ]:

def predict_label_batch_worker(batch_items, x_eval, threshold=0.5):
    import pickle
    import numpy as np

    n_rows = x_eval.shape[0]
    positive_map = {}
    best_scores = np.full(n_rows, -1.0, dtype=np.float32)
    best_labels = np.full(n_rows, -1, dtype=np.int32)

    for label_col, label_id, model_bytes in batch_items:
        model = pickle.loads(model_bytes)
        probs = model.predict_proba(x_eval)[:, 1].astype(np.float32)
        pos_idx = np.where(probs >= threshold)[0]
        if pos_idx.size > 0:
            for idx in pos_idx:
                if idx not in positive_map:
                    positive_map[idx] = [int(label_id)]
                else:
                    positive_map[idx].append(int(label_id))

        better = probs > best_scores
        best_scores[better] = probs[better]
        best_labels[better] = int(label_id)

    return {
        "positive_map": positive_map,
        "best_scores": best_scores,
        "best_labels": best_labels,
        "labels_processed": len(batch_items),
        "rows": n_rows,
    }


Executing distributed inference across available workers, aggregating thresholded outputs, applying fallback labels, and writing final prediction files.

In [ ]:

THRESHOLD = 0.5
PREDICTION_CSV = ROOT / "milestone3_predictions.csv"
PREDICTION_PARQUET = ROOT / "milestone3_predictions.parquet"

n_workers = len(active_workers)
n_rows = X_test_sparse.shape[0]

# Split labels into one batch per active worker
batch_arrays = np.array_split(np.array(model_payload_items, dtype=object), n_workers)
batches = [arr.tolist() for arr in batch_arrays if len(arr) > 0]

row_positive_labels = [[] for _ in range(n_rows)]
global_best_scores = np.full(n_rows, -1.0, dtype=np.float32)
global_best_labels = np.full(n_rows, -1, dtype=np.int32)

futures = []
inference_start = time.time()

for i, batch in enumerate(batches):
    worker_addr = active_workers[i % n_workers]
    fut = client.submit(
        predict_label_batch_worker,
        batch,
        X_test_ref,
        THRESHOLD,
        workers=[worker_addr],
        pure=False,
    )
    futures.append(fut)

print(f"Dispatched {len(futures)} distributed inference tasks.")

labels_processed_total = 0
for fut in as_completed(futures):
    out = fut.result()
    labels_processed_total += int(out["labels_processed"])

    pos_map = out["positive_map"]
    for row_idx, lbls in pos_map.items():
        row_positive_labels[int(row_idx)].extend(lbls)

    batch_best_scores = out["best_scores"]
    batch_best_labels = out["best_labels"]
    better = batch_best_scores > global_best_scores
    global_best_scores[better] = batch_best_scores[better]
    global_best_labels[better] = batch_best_labels[better]

inference_time_distributed = round(time.time() - inference_start, 3)

# Aggregation strategy:
# 1) thresholding per label (>= THRESHOLD)
# 2) fallback to best-probability label if row has no positive labels
predicted_labels_text = []
predicted_label_count = []
empty_before_fallback = 0

for i in range(n_rows):
    labels = sorted(set(row_positive_labels[i]))
    if len(labels) == 0:
        empty_before_fallback += 1
        if global_best_labels[i] >= 0:
            labels = [int(global_best_labels[i])]

    predicted_labels_text.append(" ".join([f"{lbl}:1" for lbl in labels]))
    predicted_label_count.append(len(labels))

predictions_df = pd.DataFrame({
    "row_id": row_ids,
    "predicted_labels": predicted_labels_text,
    "predicted_label_count": predicted_label_count,
})

predictions_df.to_csv(PREDICTION_CSV, index=False)

parquet_written = False
try:
    predictions_df.to_parquet(PREDICTION_PARQUET, index=False)
    parquet_written = True
except Exception as e:
    print(f"Parquet write skipped: {e}")

print("Distributed inference complete.")
print(f"Labels processed               : {labels_processed_total}")
print(f"Test rows predicted            : {len(predictions_df)}")
print(f"Rows empty before fallback     : {empty_before_fallback}")
print(f"Distributed inference time (s) : {inference_time_distributed}")
print(f"Predictions CSV saved          : {PREDICTION_CSV}")
if parquet_written:
    print(f"Predictions Parquet saved      : {PREDICTION_PARQUET}")

display(predictions_df.head(10))


Benchmarking sequential and distributed inference to measure runtime, speedup, and parallel efficiency across available worker counts.

In [ ]:

BENCHMARK_LABEL_COUNT = min(300, len(model_payload_items))
benchmark_items = model_payload_items[:BENCHMARK_LABEL_COUNT]


def benchmark_sequential(items, x_eval, threshold=0.5):
    start = time.time()
    positive_total = 0
    for _, _, model_bytes in items:
        model = pickle.loads(model_bytes)
        probs = model.predict_proba(x_eval)[:, 1]
        positive_total += int((probs >= threshold).sum())
    return round(time.time() - start, 3), positive_total


def benchmark_worker_batch(items, x_eval, threshold=0.5):
    import pickle
    import numpy as np
    positive_total = 0
    for _, _, model_bytes in items:
        model = pickle.loads(model_bytes)
        probs = model.predict_proba(x_eval)[:, 1]
        positive_total += int((probs >= threshold).sum())
    return positive_total


def split_even(items, n):
    n = max(1, int(n))
    k, m = divmod(len(items), n)
    out = []
    start = 0
    for i in range(n):
        end = start + k + (1 if i < m else 0)
        if start < end:
            out.append(items[start:end])
        start = end
    return out


seq_time, seq_pos = benchmark_sequential(benchmark_items, X_test_sparse, THRESHOLD)
print(f"Sequential benchmark labels: {BENCHMARK_LABEL_COUNT}")
print(f"Sequential time (s): {seq_time}")

available_n = len(active_workers)
target_counts = [1, 2, 4, 8]
benchmark_rows = []

for n in target_counts:
    if n > available_n:
        print(f"Skipping worker_count={n}; only {available_n} workers available.")
        continue

    selected_workers = active_workers[:n]
    parts = split_even(benchmark_items, n)

    t0 = time.time()
    futs = []
    for i, part in enumerate(parts):
        fut = client.submit(
            benchmark_worker_batch,
            part,
            X_test_ref,
            THRESHOLD,
            workers=[selected_workers[i % n]],
            pure=False,
        )
        futs.append(fut)

    dist_pos = 0
    for fut in as_completed(futs):
        dist_pos += int(fut.result())

    dist_time = round(time.time() - t0, 3)
    speedup = (seq_time / dist_time) if dist_time > 0 else np.nan
    efficiency = (speedup / n) if n > 0 else np.nan

    benchmark_rows.append({
        "worker_count": n,
        "seq_time_sec": seq_time,
        "dist_time_sec": dist_time,
        "speedup": speedup,
        "efficiency": efficiency,
        "seq_positive_count": seq_pos,
        "dist_positive_count": dist_pos,
    })

benchmark_df = pd.DataFrame(benchmark_rows).sort_values("worker_count")
BENCHMARK_CSV = ROOT / "milestone3_benchmark.csv"
benchmark_df.to_csv(BENCHMARK_CSV, index=False)

print(f"Benchmark results saved: {BENCHMARK_CSV}")
display(benchmark_df)

if not benchmark_df.empty:
    plt.figure(figsize=(8, 4))
    plt.plot(benchmark_df["worker_count"], benchmark_df["speedup"], marker="o")
    plt.title("Milestone 3 Speedup vs Worker Count")
    plt.xlabel("Worker Count")
    plt.ylabel("Speedup")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("milestone3_speedup.png", dpi=150)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(benchmark_df["worker_count"], benchmark_df["efficiency"], marker="o")
    plt.title("Milestone 3 Parallel Efficiency vs Worker Count")
    plt.xlabel("Worker Count")
    plt.ylabel("Efficiency")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("milestone3_efficiency.png", dpi=150)
    plt.show()


Saving milestone 3 manifests and summary metrics while closing the distributed client cleanly after prediction and benchmarking.

In [ ]:

manifest_m3 = {
    "milestone": 3,
    "test_rows": int(len(predictions_df)),
    "trained_models_used": int(len(model_payload_items)),
    "threshold": float(THRESHOLD),
    "distributed_inference_time_sec": float(inference_time_distributed),
    "rows_without_positive_before_fallback": int(empty_before_fallback),
    "prediction_csv": str(PREDICTION_CSV),
    "prediction_parquet": str(PREDICTION_PARQUET),
    "benchmark_csv": str(BENCHMARK_CSV),
    "speedup_plot": "milestone3_speedup.png",
    "efficiency_plot": "milestone3_efficiency.png",
    "scheduler": f"tcp://{SCHEDULER_IP}:{SCHEDULER_PORT}",
    "worker_count_used": int(len(active_workers)),
}

with open(MANIFEST_M3, "w") as f:
    json.dump(manifest_m3, f, indent=2)

print(f"Milestone 3 manifest saved: {MANIFEST_M3}")
print("Milestone 3 pipeline complete.")

client.close()
print("Dask client closed.")
